# Sionna 0.19 Scene Builder

Portable scene builder using:
- **AWS Terrain Tiles** (elevation-tiles-prod, GeoTIFF, zoom 14 ≈ 2.4 m/px) — no spikes, global coverage, free, anonymous
- **OSM via osmnx** — buildings + roads + water + vegetation
- **Approach from sionna-large-radio-maps** (NVIDIA) adapted for Sionna 0.19 / no drjit dependency

### What this notebook does
1. Downloads AWS elevation tiles for the scene bbox → builds a clean heightmap
2. Downloads OSM buildings → extrudes each building with correct `base_z` from heightmap
3. Builds terrain PLY mesh with real DEM elevation
4. Writes `scene.xml` (Mitsuba 2.1.0) compatible with Sionna 0.19
5. Saves everything to `BASE_DIR/scene/` ready for the main simulation notebook

### Run order
`CELL 0 → CELL 1 → CELL 2 → CELL 3 → CELL 4 → CELL 5 → CELL 6`

In [ ]:
# ============================================================
# CELL 0 — CONFIG  (edit this cell only)
# ============================================================
import os

# ── Scene bbox (WGS84) ────────────────────────────────────────────────────
# Tight bbox: TX + first 1200 RX (915.95 MHz) + 900m margin → 10.1×7.3 km = 74 km²
SCENE_WEST   = -1.260093
SCENE_EAST   = -1.129307
SCENE_SOUTH  =  52.945798
SCENE_NORTH  =  52.998702

# ── Output directory ─────────────────────────────────────────────────────
BASE_DIR  = os.path.expanduser('~/Documents/FYP2026/nottingham900')
SCENE_DIR = os.path.join(BASE_DIR, 'scene')
MESH_DIR  = os.path.join(SCENE_DIR, 'meshes')
os.makedirs(MESH_DIR, exist_ok=True)

# ── Coordinate system ────────────────────────────────────────────────────
UTM_EPSG  = 32630   # UTM zone 30N (covers UK)

# ── AWS tile zoom level ──────────────────────────────────────────────────
# z=14 → ~2.4 m/px at UK latitude (512×512 px tiles)
# z=13 → ~4.8 m/px  (faster download, less detail)
TILE_ZOOM  = 14

# ── Terrain mode ─────────────────────────────────────────────────────────────
# FLAT_TERRAIN = True  → fast flat z=0 plane, skip CELL 2 and CELL 2b
# FLAT_TERRAIN = False → real terrain from EA LiDAR DTM (auto-downloaded in CELL 2b)
#                        or AWS DEM tiles (CELL 2) — EA LiDAR recommended for UK
FLAT_TERRAIN = True   # ← change to False for realistic terrain (run CELL 2b first)

# ── Terrain source (only used when FLAT_TERRAIN=False) ────────────────────────
# 'ea_lidar' = Environment Agency 1m DTM (auto-download, UK only, most accurate)
# 'aws_dem'  = AWS elevation tiles 2.4m DSM (global, less accurate)
TERRAIN_SOURCE = 'ea_lidar'

# EA LiDAR output path (auto-set, edit only if needed)
EA_DTM_TIFF = os.path.join(BASE_DIR, 'ea_dtm_1m.tif')

# ── Terrain mesh resolution ───────────────────────────────────────────────
# Number of grid points per axis for the terrain PLY.
# 500 → 500×500 = 250k verts, ~500k triangles (~10 MB PLY) — recommended
# 200 → 200×200 = 40k  verts, ~80k triangles  (~1.5 MB PLY) — fast
TERRAIN_GRID_N = 500

# ── Building parameters ───────────────────────────────────────────────────
MIN_BUILDING_AREA_M2  = 30.0   # skip footprints smaller than this
CITY_MIN_HEIGHT_M     =  2.0   # clamp building height to at least this
CITY_MAX_HEIGHT_M     = 40.0   # clamp building height to at most this
HEIGHT_PER_LEVEL_M    =  3.5   # used when only building:levels tag present
DEFAULT_HEIGHT_M      =  8.0   # fallback when no height/levels tag

# ── OSM extra features ────────────────────────────────────────────────────
INCLUDE_ROADS       = True
INCLUDE_WATER       = True
INCLUDE_VEGETATION  = False   # polygon veg — adds clutter for macro-cell sim

# ── Exclude small/irrelevant building types ───────────────────────────────
EXCLUDE_BUILDING_TYPES = {
    'garage','garages','carport','shed','hut','roof','canopy',
    'kiosk','bicycle_parking','service','greenhouse','barn',
    'stable','sty','storage_tank','container','tent',
    'grandstand','shelter','utility','gatehouse',
}

print('Config loaded.')
print(f'  Scene bbox  : lon [{SCENE_WEST}, {SCENE_EAST}]')
print(f'                lat [{SCENE_SOUTH}, {SCENE_NORTH}]')
print(f'  Output      : {SCENE_DIR}')
print(f'  Tile zoom   : {TILE_ZOOM}  (~{156543.03 * np.cos(np.radians((SCENE_SOUTH+SCENE_NORTH)/2)) / 2**TILE_ZOOM:.1f} m/px)' if 'np' in dir() else f'  Tile zoom   : {TILE_ZOOM}')

In [ ]:
# ============================================================
# CELL 1 — IMPORTS & DEPENDENCIES
# ============================================================
import os, math, time, json, struct, warnings
import numpy as np
import requests
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyproj import Transformer
import shapely.geometry as sg
import shapely.ops as so
from shapely.geometry import Polygon, MultiPolygon, box

# Optional: rasterio for GeoTIFF reading
try:
    import rasterio
    from rasterio.transform import from_bounds
    _HAS_RASTERIO = True
except ImportError:
    _HAS_RASTERIO = False
    print('⚠  rasterio not found — falling back to PIL for GeoTIFF tiles')

# PIL for fallback
try:
    from PIL import Image
    _HAS_PIL = True
except ImportError:
    _HAS_PIL = False

# osmnx for OSM data
try:
    import osmnx as ox
    ox.settings.use_cache = True
    ox.settings.log_console = False
    _HAS_OSMNX = True
except ImportError:
    _HAS_OSMNX = False
    print('⚠  osmnx not found — install with: pip install osmnx')

# trimesh for PLY export
try:
    import trimesh
    _HAS_TRIMESH = True
except ImportError:
    _HAS_TRIMESH = False
    print('⚠  trimesh not found — install with: pip install trimesh')

# Coordinate transformers
to_utm   = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
to_wgs84 = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)

# Scene bbox in UTM
sw_utm = to_utm.transform(SCENE_WEST,  SCENE_SOUTH)
ne_utm = to_utm.transform(SCENE_EAST,  SCENE_NORTH)
center_utm = ((sw_utm[0]+ne_utm[0])/2, (sw_utm[1]+ne_utm[1])/2)
center_lon, center_lat = to_wgs84.transform(*center_utm)

print(f'UTM SW : {sw_utm[0]:.1f}, {sw_utm[1]:.1f}')
print(f'UTM NE : {ne_utm[0]:.1f}, {ne_utm[1]:.1f}')
print(f'Center : ({center_lon:.5f}, {center_lat:.5f})')
print(f'Size   : {(ne_utm[0]-sw_utm[0])/1000:.2f} km × {(ne_utm[1]-sw_utm[1])/1000:.2f} km')

In [ ]:
# ============================================================
# CELL 2 — AWS ELEVATION TILES → HEIGHTMAP
# ============================================================
# Downloads GeoTIFF tiles from the public AWS elevation-tiles-prod bucket
# (same source used by Mapzen/Terrarium, Cesium, sionna-large-radio-maps).
# No credentials needed — anonymous public access.
# URL: s3://elevation-tiles-prod/geotiff/{z}/{x}/{y}.tif
# Values are direct metres ASL (float32 GeoTIFF, no conversion formula).

AWS_BASE = 'https://s3.amazonaws.com/elevation-tiles-prod/geotiff'

def _lon2tile(lon, z):
    return int(math.floor((lon + 180) / 360 * 2**z))

def _lat2tile(lat, z):
    lat_r = math.radians(lat)
    return int(math.floor((1 - math.log(math.tan(lat_r) + 1/math.cos(lat_r)) / math.pi) / 2 * 2**z))

def _tile2lon(x, z):
    return x / 2**z * 360 - 180

def _tile2lat(y, z):
    n = math.pi - 2 * math.pi * y / 2**z
    return math.degrees(math.atan(math.sinh(n)))

def _download_tile(z, x, y, retries=3):
    url = f'{AWS_BASE}/{z}/{x}/{y}.tif'
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            buf = BytesIO(r.content)
            if _HAS_RASTERIO:
                with rasterio.open(buf) as ds:
                    data = ds.read(1).astype(np.float32)
            elif _HAS_PIL:
                # PIL may not read float GeoTIFF correctly — warn
                img = Image.open(buf)
                data = np.array(img, dtype=np.float32)
            else:
                raise RuntimeError('Need rasterio or PIL to read GeoTIFF tiles')
            # Resample to 512×512 if needed
            if data.shape != (512, 512):
                from scipy.ndimage import zoom as nd_zoom
                fx = 512 / data.shape[0]; fy = 512 / data.shape[1]
                data = nd_zoom(data, (fx, fy), order=1).astype(np.float32)
            return data
        except Exception as e:
            if attempt == retries - 1:
                print(f'  ⚠ tile {z}/{x}/{y} failed: {e} — using zeros')
                return np.zeros((512, 512), dtype=np.float32)
            time.sleep(2 ** attempt)

# Compute tile range for scene bbox
x0 = _lon2tile(SCENE_WEST,  TILE_ZOOM)
x1 = _lon2tile(SCENE_EAST,  TILE_ZOOM)
y0 = _lat2tile(SCENE_NORTH, TILE_ZOOM)  # north = smaller y in tile coords
y1 = _lat2tile(SCENE_SOUTH, TILE_ZOOM)
n_cols = x1 - x0 + 1
n_rows = y1 - y0 + 1
print(f'Tile range : x=[{x0},{x1}] y=[{y0},{y1}]  →  {n_cols}×{n_rows} = {n_cols*n_rows} tiles')

# Download in parallel
tile_mosaic = np.zeros((n_rows * 512, n_cols * 512), dtype=np.float32)
jobs = [(TILE_ZOOM, x0+col, y0+row, col, row)
        for row in range(n_rows) for col in range(n_cols)]

print(f'Downloading {len(jobs)} tiles ...')
t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {ex.submit(_download_tile, z, x, y): (col, row)
               for z, x, y, col, row in jobs}
    for i, fut in enumerate(as_completed(futures)):
        col, row = futures[fut]
        tile_mosaic[row*512:(row+1)*512, col*512:(col+1)*512] = fut.result()
        if (i+1) % max(1, len(jobs)//4) == 0:
            print(f'  {i+1}/{len(jobs)} tiles done')

print(f'Download done in {time.time()-t0:.1f}s')

# Extent of the mosaic in WGS84
_map_min_lon = _tile2lon(x0,   TILE_ZOOM)
_map_max_lon = _tile2lon(x1+1, TILE_ZOOM)
_map_max_lat = _tile2lat(y0,   TILE_ZOOM)   # y0 is north
_map_min_lat = _tile2lat(y1+1, TILE_ZOOM)   # y1+1 is south
_mosaic_h, _mosaic_w = tile_mosaic.shape

print(f'Mosaic size : {_mosaic_w}×{_mosaic_h} px')
print(f'Mosaic lon  : [{_map_min_lon:.5f}, {_map_max_lon:.5f}]')
print(f'Mosaic lat  : [{_map_min_lat:.5f}, {_map_max_lat:.5f}]')
print(f'Elevation   : [{tile_mosaic.min():.1f}, {tile_mosaic.max():.1f}] m ASL')

def height_from_wgs84(lon, lat):
    """Bilinear interpolation from mosaic. Returns elevation in metres ASL."""
    u = (lon - _map_min_lon) / (_map_max_lon - _map_min_lon) * (_mosaic_w - 1)
    v = (1 - (lat - _map_min_lat) / (_map_max_lat - _map_min_lat)) * (_mosaic_h - 1)
    u = np.clip(u, 0, _mosaic_w - 1)
    v = np.clip(v, 0, _mosaic_h - 1)
    x0i, y0i = int(np.floor(u)), int(np.floor(v))
    x1i, y1i = min(x0i+1, _mosaic_w-1), min(y0i+1, _mosaic_h-1)
    fx, fy = u - x0i, v - y0i
    h = (tile_mosaic[y0i, x0i] * (1-fx) * (1-fy)
       + tile_mosaic[y0i, x1i] *    fx  * (1-fy)
       + tile_mosaic[y1i, x0i] * (1-fx) *    fy
       + tile_mosaic[y1i, x1i] *    fx  *    fy)
    return float(h)

def height_from_utm(easting, northing):
    """Height at UTM coordinates."""
    lon, lat = to_wgs84.transform(easting, northing)
    return height_from_wgs84(lon, lat)

# Scene centre elevation = local z=0 reference
origin_elev_asl = height_from_wgs84(center_lon, center_lat)
print(f'\nScene centre elevation : {origin_elev_asl:.2f} m ASL  (= local z=0)')

def local_z(lon, lat):
    """Returns local z in metres relative to scene centre elevation."""
    return height_from_wgs84(lon, lat) - origin_elev_asl

In [ ]:
# ============================================================
# CELL 2b — EA LiDAR DTM Auto-Download  (skip if FLAT_TERRAIN=True)
# ============================================================
# Downloads Environment Agency 1m Composite DTM for scene bbox.
# Source: EA WCS service (free, no credentials, England only).
# CRS: EPSG:27700 (British National Grid) for request, output GeoTIFF.
# Skip this cell entirely when FLAT_TERRAIN=True.
# ============================================================

if globals().get('FLAT_TERRAIN', True):
    print('FLAT_TERRAIN=True — skipping EA LiDAR download.')
    print('Set FLAT_TERRAIN=False in CELL 0 and re-run this cell for real terrain.')
else:
    import requests, os
    from pyproj import Transformer

    print('=' * 60)
    print('CELL 2b — EA LiDAR DTM Download (1m resolution)')
    print('=' * 60)

    if os.path.exists(EA_DTM_TIFF):
        print(f'Already downloaded: {EA_DTM_TIFF}')
        print('Delete the file and re-run to force re-download.')
    else:
        # Convert scene bbox WGS84 → BNG (EPSG:27700) for EA WCS request
        _wgs_to_bng = Transformer.from_crs('EPSG:4326', 'EPSG:27700', always_xy=True)
        _e_min, _n_min = _wgs_to_bng.transform(SCENE_WEST,  SCENE_SOUTH)
        _e_max, _n_max = _wgs_to_bng.transform(SCENE_EAST,  SCENE_NORTH)

        # Add 200m buffer so terrain edge doesn't clip buildings
        _buf = 200
        _e_min -= _buf; _n_min -= _buf
        _e_max += _buf; _n_max += _buf

        print(f'BNG bbox: E[{_e_min:.0f}, {_e_max:.0f}]  N[{_n_min:.0f}, {_n_max:.0f}]')
        print(f'Area    : {(_e_max-_e_min)/1000:.1f} km × {(_n_max-_n_min)/1000:.1f} km')

        # EA WCS endpoint — Composite DTM 1m
        _WCS_URL = (
            'https://environment.data.gov.uk/spatialdata/lidar-composite-dtm-1m/wcs'
            '?SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage'
            '&COVERAGEID=LIDAR_Composite_DTM_1m'
            f'&SUBSET=E,http://www.opengis.net/def/crs/EPSG/0/27700({_e_min:.0f},{_e_max:.0f})'
            f'&SUBSET=N,http://www.opengis.net/def/crs/EPSG/0/27700({_n_min:.0f},{_n_max:.0f})'
            '&FORMAT=image/tiff'
        )

        print(f'Downloading EA LiDAR DTM ...')
        print(f'URL: {_WCS_URL[:100]}...')

        try:
            _r = requests.get(_WCS_URL, timeout=120, stream=True)
            _r.raise_for_status()
            os.makedirs(os.path.dirname(EA_DTM_TIFF), exist_ok=True)
            _total = 0
            with open(EA_DTM_TIFF, 'wb') as _f:
                for _chunk in _r.iter_content(chunk_size=1024*1024):
                    _f.write(_chunk)
                    _total += len(_chunk)
                    print(f'  {_total//1024//1024} MB downloaded ...', end='\r')
            print(f'\nSaved: {EA_DTM_TIFF}  ({os.path.getsize(EA_DTM_TIFF)//1024//1024} MB)')

            # Quick verify with rasterio
            if _HAS_RASTERIO:
                import rasterio as _rio
                with _rio.open(EA_DTM_TIFF) as _ds:
                    print(f'CRS     : {_ds.crs}')
                    print(f'Shape   : {_ds.height} × {_ds.width} px')
                    print(f'Res     : {_ds.res[0]:.1f} m/px')
                    _data = _ds.read(1)
                    print(f'Z range : {float(_data.min()):.1f} – {float(_data.max()):.1f} m ASL')
            print('\n✓ EA LiDAR DTM ready. Now run CELL 3 to build terrain mesh.')

        except requests.exceptions.HTTPError as _e:
            print(f'HTTP error: {_e}')
            print('The EA WCS may be temporarily unavailable. Try again later.')
            print('Alternative: download manually from:')
            print('  https://environment.data.gov.uk/DefraDataDownload/?Mode=survey')
            print('  Select: LIDAR Composite DTM → 1m → your area → GeoTIFF')
            print(f'  Save to: {EA_DTM_TIFF}')
        except Exception as _e:
            print(f'Download failed: {_e}')
            print(f'Manual download URL:')
            print('  https://environment.data.gov.uk/DefraDataDownload/?Mode=survey')


In [ ]:
# ============================================================
# CELL 3 — BUILD TERRAIN PLY
# ============================================================
# FLAT_TERRAIN=True  : flat z=0 plane — fast, no DEM needed
# FLAT_TERRAIN=False : DEM elevation from AWS tiles (run CELL 2 first)
# ============================================================
import struct, os
import numpy as np

N = TERRAIN_GRID_N

if FLAT_TERRAIN:
    print(f'Building FLAT terrain mesh: {N}×{N} grid ...')
    x_span = (SCENE_EAST  - SCENE_WEST)  * 111000 * np.cos(np.radians((SCENE_SOUTH+SCENE_NORTH)/2))
    y_span = (SCENE_NORTH - SCENE_SOUTH) * 111000
    xs = np.linspace(-x_span/2, x_span/2, N, dtype=np.float32)
    ys = np.linspace(-y_span/2, y_span/2, N, dtype=np.float32)
    XX, YY = np.meshgrid(xs, ys, indexing='ij')
    ZZ = np.zeros((N, N), dtype=np.float32)
    origin_elev_asl = 0.0
    def local_z(lon, lat): return 0.0
else:
    _src = globals().get('TERRAIN_SOURCE', 'aws_dem')
    if _src == 'ea_lidar' and os.path.exists(globals().get('EA_DTM_TIFF','')):
        print(f'Building terrain mesh from EA LiDAR DTM: {N}×{N} grid ...')
        import rasterio as _rio
        from pyproj import Transformer as _Tr
        _bng_to_utm = _Tr.from_crs('EPSG:27700', f'EPSG:{UTM_EPSG}', always_xy=True)
        _utm_to_bng = _Tr.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:27700', always_xy=True)
        _ds = _rio.open(EA_DTM_TIFF)
        _dem_data = _ds.read(1).astype(np.float32)
        _dem_tf   = _ds.transform
        _dem_nd   = _ds.nodata

        # scene origin elevation from EA DTM
        _ox_bng, _oy_bng = _utm_to_bng.transform(center_utm[0], center_utm[1])
        _oc, _or = ~_dem_tf * (_ox_bng, _oy_bng)
        _or, _oc = int(_or), int(_oc)
        if 0 <= _or < _dem_data.shape[0] and 0 <= _oc < _dem_data.shape[1]:
            origin_elev_asl = float(_dem_data[_or, _oc])
        else:
            origin_elev_asl = 0.0
        print(f'  Origin elevation: {origin_elev_asl:.1f} m ASL')

        def _ea_z(utm_x, utm_y):
            bx, by = _utm_to_bng.transform(utm_x, utm_y)
            col_f, row_f = ~_dem_tf * (bx, by)
            r, c = int(row_f), int(col_f)
            H, W = _dem_data.shape
            if 0 <= r < H-1 and 0 <= c < W-1:
                dr, dc = row_f-r, col_f-c
                z = ((1-dr)*(1-dc)*_dem_data[r,c] + (1-dr)*dc*_dem_data[r,c+1] +
                     dr*(1-dc)*_dem_data[r+1,c] + dr*dc*_dem_data[r+1,c+1])
                if _dem_nd is None or not np.isclose(float(z), _dem_nd):
                    return float(z)
            return origin_elev_asl

        x_span = (SCENE_EAST-SCENE_WEST)*111000*np.cos(np.radians((SCENE_SOUTH+SCENE_NORTH)/2))
        y_span = (SCENE_NORTH-SCENE_SOUTH)*111000
        xs = np.linspace(-x_span/2, x_span/2, N, dtype=np.float32)
        ys = np.linspace(-y_span/2, y_span/2, N, dtype=np.float32)
        XX, YY = np.meshgrid(xs, ys, indexing='ij')
        ZZ = np.zeros((N, N), dtype=np.float32)
        import time as _t; t0 = _t.time()
        for i in range(N):
            for j in range(N):
                ZZ[i,j] = _ea_z(center_utm[0]+XX[i,j], center_utm[1]+YY[i,j]) - origin_elev_asl
            if (i+1) % max(1,N//5)==0:
                print(f'  row {i+1}/{N}  ({_t.time()-t0:.0f}s)')
        def local_z(lon, lat):
            ux,uy = to_utm.transform(lon,lat)
            return _ea_z(ux,uy) - origin_elev_asl
    else:
        print(f'Building terrain mesh from AWS DEM: {N}×{N} grid ...')
        x_span = ne_utm[0] - sw_utm[0]
        y_span = ne_utm[1] - sw_utm[1]
        xs = np.linspace(-x_span/2, x_span/2, N, dtype=np.float32)
        ys = np.linspace(-y_span/2, y_span/2, N, dtype=np.float32)
        XX, YY = np.meshgrid(xs, ys, indexing='ij')
        ZZ = np.zeros((N, N), dtype=np.float32)
        import time as _t; t0 = _t.time()
        for i in range(N):
            for j in range(N):
                utm_x = center_utm[0] + XX[i, j]
                utm_y = center_utm[1] + YY[i, j]
                ZZ[i, j] = height_from_utm(utm_x, utm_y) - origin_elev_asl
            if (i+1) % max(1, N//5) == 0:
                print(f'  row {i+1}/{N}  ({_t.time()-t0:.0f}s)')

print(f'Terrain Z range: [{ZZ.min():.1f}, {ZZ.max():.1f}] m')

verts = np.stack([XX.ravel(), YY.ravel(), ZZ.ravel()], axis=1).astype(np.float32)
faces = []
for i in range(N-1):
    for j in range(N-1):
        a = i*N+j; b = a+1; c = a+N; d = c+1
        faces.append([a,b,d]); faces.append([a,d,c])
faces = np.array(faces, dtype=np.int32)

# Write binary PLY
ply_path = os.path.join(MESH_DIR, 'terrain.ply')
with open(ply_path, 'wb') as f:
    hdr = (f'ply\nformat binary_little_endian 1.0\n'
           f'element vertex {len(verts)}\nproperty float x\nproperty float y\nproperty float z\n'
           f'element face {len(faces)}\nproperty list uchar int vertex_indices\nend_header\n')
    f.write(hdr.encode())
    f.write(verts.tobytes())
    for fc in faces:
        f.write(struct.pack('<B3i', 3, *fc))

kb = os.path.getsize(ply_path)//1024
print(f'terrain.ply: {len(verts):,} verts  {len(faces):,} faces  {kb} KB  → {ply_path}')

# Save origin elevation for main notebook compatibility
import json as _json
_elev_path = os.path.join(SCENE_DIR, 'origin_elev2.json')
_json.dump({'origin_elev_asl_m': float(origin_elev_asl), 'flat_terrain': FLAT_TERRAIN}, open(_elev_path, 'w'))
print(f'origin_elev2.json: {_elev_path}')


In [ ]:
# ============================================================
# CELL 4 — OSM BUILDINGS → BUILDING PLYS
# ============================================================

assert _HAS_OSMNX, 'osmnx required: pip install osmnx'

import osmnx as _ox_ver
_ox_version = tuple(int(x) for x in _ox_ver.__version__.split('.')[:2])

print(f'Downloading OSM buildings (osmnx {_ox_ver.__version__}) ...')
t0 = time.time()
try:
    if _ox_version >= (2, 0):
        # osmnx >= 2.0: bbox=(left, bottom, right, top) = (west, south, east, north)
        gdf_bld = ox.features_from_bbox(
            bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH),
            tags={'building': True}
        )
    elif _ox_version >= (1, 3):
        # osmnx 1.3–1.x: bbox=(north, south, east, west)
        gdf_bld = ox.features_from_bbox(
            bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST),
            tags={'building': True}
        )
    else:
        # osmnx < 1.3: keyword arguments
        gdf_bld = ox.features_from_bbox(
            north=SCENE_NORTH, south=SCENE_SOUTH,
            east=SCENE_EAST,   west=SCENE_WEST,
            tags={'building': True}
        )
except Exception as e:
    print(f'osmnx error: {e}')
    raise
print(f'  {len(gdf_bld)} raw building features  ({time.time()-t0:.1f}s)')

# ── Helper: wall material from OSM tags ────────────────────────────────────
def _bld_mat(row):
    mat  = str(row.get('building:material', '')).lower()
    fmat = str(row.get('building:facade:material', '')).lower()
    tag  = str(row.get('building', '')).lower()
    amen = str(row.get('amenity', '')).lower()
    shop = str(row.get('shop', '')).lower()
    off  = str(row.get('office', '')).lower()
    if 'glass' in mat or 'glass' in fmat:                return 'itu_glass'
    if 'wood'  in mat or 'timber' in mat:                return 'itu_wood'
    if 'wood'  in fmat or 'timber' in fmat:              return 'itu_wood'
    if 'brick' in mat or 'brick' in fmat:                return 'itu_brick'
    if 'stone' in mat or 'stone' in fmat:                return 'itu_brick'
    if tag in ('greenhouse','glasshouse'):               return 'itu_glass'
    if amen in ('shopping_centre','mall'):               return 'itu_glass'
    if shop in ('mall','supermarket','department_store'):return 'itu_glass'
    if off:                                              return 'itu_glass'
    if tag in ('residential','house','detached','semidetached_house',
               'semi_detached','terrace','terrace_house','bungalow',
               'farm','farmhouse','dormitory','apartments','block'):
        return 'itu_brick'
    if tag in ('industrial','warehouse','factory','shed',
               'storage_tank','silo','barn'):            return 'itu_concrete'
    if tag in ('retail','commercial','supermarket','kiosk'): return 'itu_glass'
    if tag in ('cathedral','church','chapel','mosque','temple'): return 'itu_brick'
    if tag in ('school','university','hospital','civic','public'): return 'itu_concrete'
    return 'itu_brick'   # UK default

def _roof_mat(row):
    roof_tag = str(row.get('roof:material', '')).lower()
    btag     = str(row.get('building', '')).lower()
    if any(k in roof_tag for k in ['metal','steel','zinc','aluminium','copper','tin']):
        return 'itu_metal'
    if 'glass' in roof_tag:
        return 'itu_glass'
    if any(k in roof_tag for k in ['wood','timber','thatch']):
        return 'itu_wood'
    if any(k in roof_tag for k in ['tile','concrete','slate','terracotta']):
        return 'itu_concrete'
    if btag in ('industrial','warehouse','factory','shed','barn',
                'retail','supermarket','commercial','garage','garages'):
        return 'itu_metal'
    return 'itu_concrete'   # UK residential default: concrete tile

def _bld_height(row):
    try:
        h = float(str(row.get('height','0')).replace('m','').strip())
        if h > 1:
            return np.clip(h, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M)
    except:
        pass
    try:
        lvl = float(str(row.get('building:levels','0')).strip())
        if lvl > 0:
            return np.clip(lvl * HEIGHT_PER_LEVEL_M, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M)
    except:
        pass
    return DEFAULT_HEIGHT_M

def _write_ply(verts, faces, path):
    verts = np.asarray(verts, dtype=np.float32)
    faces = np.asarray(faces, dtype=np.int32)
    if _HAS_TRIMESH:
        trimesh.Trimesh(vertices=verts, faces=faces, process=False).export(path)
        return
    with open(path, 'w') as f:
        f.write('ply\nformat ascii 1.0\n')
        f.write(f'element vertex {len(verts)}\n')
        f.write('property float x\nproperty float y\nproperty float z\n')
        f.write(f'element face {len(faces)}\n')
        f.write('property list uchar int vertex_indices\nend_header\n')
        for v in verts:
            f.write(f'{v[0]:.4f} {v[1]:.4f} {v[2]:.4f}\n')
        for fc in faces:
            f.write(f'3 {fc[0]} {fc[1]} {fc[2]}\n')

def _extrude_building(poly_utm, base_z, height):
    pts = np.array(poly_utm, dtype=np.float32)
    if len(pts) < 3:
        return None
    if np.allclose(pts[0], pts[-1]):
        pts = pts[:-1]
    n = len(pts)
    top_z = base_z + height
    bot_z = base_z

    wall_v, wall_f = [], []
    for i in range(n):
        j = (i+1) % n
        v_base = len(wall_v)
        wall_v += [
            [pts[i,0], pts[i,1], bot_z],
            [pts[j,0], pts[j,1], bot_z],
            [pts[j,0], pts[j,1], top_z],
            [pts[i,0], pts[i,1], top_z],
        ]
        wall_f += [
            [v_base,   v_base+1, v_base+2],
            [v_base,   v_base+2, v_base+3],
        ]

    roof_v = [[pts[i,0], pts[i,1], top_z] for i in range(n)]
    roof_cx = pts[:,0].mean()
    roof_cy = pts[:,1].mean()
    roof_v.append([roof_cx, roof_cy, top_z])
    centre_idx = len(roof_v) - 1
    roof_f = [[i, (i+1)%n, centre_idx] for i in range(n)]

    return (np.array(wall_v, np.float32),  np.array(wall_f,  np.int32),
            np.array(roof_v, np.float32),  np.array(roof_f,  np.int32))

# ── Process buildings ────────────────────────────────────────────────────
mat_plys = {}
n_ok = n_skip = 0

for idx, (oid, row) in enumerate(gdf_bld.iterrows()):
    geom = row.geometry
    if geom is None or geom.is_empty:
        n_skip += 1; continue

    if isinstance(geom, MultiPolygon):
        geom = max(geom.geoms, key=lambda g: g.area)
    if not isinstance(geom, Polygon):
        n_skip += 1; continue

    btag = str(row.get('building','')).lower()
    if btag in EXCLUDE_BUILDING_TYPES:
        n_skip += 1; continue

    coords_wgs = list(geom.exterior.coords)
    coords_utm = []
    for lon, lat in coords_wgs:
        ex, ny = to_utm.transform(lon, lat)
        coords_utm.append((ex - center_utm[0], ny - center_utm[1]))

    area_m2 = sg.Polygon(coords_utm).area
    if area_m2 < MIN_BUILDING_AREA_M2:
        n_skip += 1; continue

    c_lon, c_lat = geom.centroid.x, geom.centroid.y
    base_z_local = local_z(c_lon, c_lat)
    h = _bld_height(row)

    result = _extrude_building(coords_utm, base_z_local, h)
    if result is None:
        n_skip += 1; continue
    wv, wf, rv, rf = result

    w_mat = _bld_mat(row)
    r_mat = _roof_mat(row)

    wall_name = f'bld_{n_ok:05d}_wall.ply'
    roof_name = f'bld_{n_ok:05d}_roof.ply'
    _write_ply(wv, wf, os.path.join(MESH_DIR, wall_name))
    _write_ply(rv, rf, os.path.join(MESH_DIR, roof_name))

    mat_plys.setdefault(w_mat, []).append(('meshes/' + wall_name, 'wall'))
    mat_plys.setdefault(r_mat, []).append(('meshes/' + roof_name, 'roof'))

    n_ok += 1
    if n_ok % 500 == 0:
        print(f'  {n_ok} buildings written ...')

print(f'\nBuildings : {n_ok} exported, {n_skip} skipped')
print(f'Materials : {list(mat_plys.keys())}')

In [ ]:
# ============================================================
# CELL 5 — WRITE SCENE.XML  (Mitsuba 2.1.0 / Sionna 0.19)
# ============================================================
# ITU-R P.2040-2 material definitions as used in Sionna 0.19.
# Each shape references a PLY file and a material by name.

ITU_MATERIALS = {
    # name          : (relative_permittivity, conductivity_S_m)
    'itu_concrete'  : (5.31,  0.092),
    'itu_brick'     : (3.75,  0.038),
    'itu_glass'     : (6.27,  0.000),
    'itu_wood'      : (1.99,  0.000),
    'itu_metal'     : (1.00, 1.0e7 ),
    'itu_asphalt'   : (2.56,  0.000),
    'itu_wet_ground': (30.0,  0.020),
}
TERRAIN_MATERIAL = 'itu_wet_ground'

# Collect all materials actually used
used_mats = set(mat_plys.keys()) | {TERRAIN_MATERIAL}

lines = []
lines.append('<?xml version="1.0" encoding="utf-8"?>')
lines.append('<scene version="2.1.0">')
lines.append('')
lines.append('  <!-- ── ITU-R P.2040-2 Materials ────────────────────── -->')

for mat_name, (eps, sigma) in ITU_MATERIALS.items():
    if mat_name not in used_mats:
        continue
    lines.append(f'  <bsdf type="conductor" id="{mat_name}">')
    lines.append(f'    <float name="eta" value="{eps}"/>')
    lines.append(f'    <float name="k"   value="{sigma}"/>')
    lines.append(f'  </bsdf>')
    lines.append('')

lines.append('  <!-- ── Terrain ─────────────────────────────────────── -->')
lines.append('  <shape type="ply">')
lines.append('    <string name="filename" value="meshes/terrain.ply"/>')
lines.append('    <ref id="{mat}" name="bsdf"/>'.replace('{mat}', TERRAIN_MATERIAL))
lines.append('  </shape>')
lines.append('')

lines.append('  <!-- ── Buildings ───────────────────────────────────── -->')
for mat_name, ply_list in sorted(mat_plys.items()):
    for ply_path, role in ply_list:
        lines.append(f'  <shape type="ply">')
        lines.append(f'    <string name="filename" value="{ply_path}"/>')
        lines.append(f'    <ref id="{mat_name}" name="bsdf"/>')
        lines.append(f'  </shape>')

lines.append('')
lines.append('</scene>')

scene_xml = os.path.join(SCENE_DIR, 'scene.xml')
with open(scene_xml, 'w') as f:
    f.write('\n'.join(lines))

print(f'Wrote: {scene_xml}')
print(f'  Materials : {len(used_mats)}')
total_shapes = 1 + sum(len(v) for v in mat_plys.values())
print(f'  Shapes    : {total_shapes}  (1 terrain + {total_shapes-1} building parts)')

# Save scene metadata for main notebook
meta = {
    'scene_center_lon'  : center_lon,
    'scene_center_lat'  : center_lat,
    'origin_elev_asl_m' : origin_elev_asl,
    'utm_epsg'          : UTM_EPSG,
    'bbox'              : {'west': SCENE_WEST, 'east': SCENE_EAST,
                           'south': SCENE_SOUTH, 'north': SCENE_NORTH},
    'n_buildings'       : n_ok,
    'terrain_grid_n'    : TERRAIN_GRID_N,
    'tile_zoom'         : TILE_ZOOM,
}
params_json = os.path.join(BASE_DIR, 'scene_parameters.json')
with open(params_json, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'  Metadata  : {params_json}')

In [ ]:
# ============================================================
# CELL 6 — VERIFY SCENE
# ============================================================
# Quick sanity checks before handing the scene to the main notebook.

import glob as glob_mod

ply_files = sorted(glob_mod.glob(os.path.join(MESH_DIR, '*.ply')))
total_kb = sum(os.path.getsize(p) for p in ply_files) / 1024

print('=' * 60)
print('SCENE VERIFICATION')
print('=' * 60)
print(f'PLY files   : {len(ply_files)}')
print(f'Total size  : {total_kb/1024:.1f} MB')
print()

print(f'scene.xml   : {os.path.getsize(scene_xml)/1024:.0f} KB')

# Load with Mitsuba to verify (requires Sionna env)
try:
    import mitsuba as mi
    mi.set_variant('scalar_rgb')
    scene_mi = mi.load_file(scene_xml)
    bbox = scene_mi.bbox()
    print()
    print(f'Mitsuba load: OK')
    print(f'  BBox X    : [{float(bbox.min[0]):.1f}, {float(bbox.max[0]):.1f}] m')
    print(f'  BBox Y    : [{float(bbox.min[1]):.1f}, {float(bbox.max[1]):.1f}] m')
    print(f'  BBox Z    : [{float(bbox.min[2]):.1f}, {float(bbox.max[2]):.1f}] m')
except Exception as e:
    print(f'Mitsuba load: {e}')

print()
print('Scene metadata (scene_parameters.json):')
with open(params_json) as f:
    print(json.dumps(json.load(f), indent=2))

print()
print('DONE — scene ready for sionna019_main_simulation.ipynb')
print(f'Set BASE_DIR = "{BASE_DIR}" in Cell 0c of the main notebook.')

## CELL 7 — 2D Scene Map: OSM Buildings + Receiver Locations + RSSI Heatmap

In [ ]:
# ============================================================
# CELL 7 — 2D MAP: OSM BUILDINGS + RX DOTS + RSSI HEATMAP
# ============================================================
# Requires: matplotlib, pandas, scipy
# Run after CELL 4 (buildings downloaded into gdf_bld)
# Set OFCOM_CSV below to your drive-test CSV path.
# ============================================================

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.collections import PatchCollection
import pandas as pd
import numpy as np
import re
import warnings

# ── Config ────────────────────────────────────────────────────────────────
OFCOM_CSV   = os.path.join(BASE_DIR, 'nottingham915.csv')   # path to drive-test CSV
MAX_RX_PLOT = None    # set to e.g. 1200 to plot only first N rows; None = all
PLOT_HEATMAP = True   # True = RSSI colour scatter; False = plain blue dots
SAVE_FIG    = True    # save PNG to BASE_DIR
FIG_DPI     = 150

# ── Parse CSV: TX coords + RX lat/lon/RSSI ───────────────────────────────
_tx_lat = _tx_lon = None
_rx_rows = []

if not os.path.exists(OFCOM_CSV):
    print(f'CSV not found: {OFCOM_CSV}')
    print('Set OFCOM_CSV path above. Buildings will still be plotted.')
else:
    with open(OFCOM_CSV, 'r', encoding='utf-8', errors='replace') as _f:
        _lines = _f.readlines()

    for _line in _lines[:30]:
        if 'Site latitude' in _line and _tx_lat is None:
            _m = re.search(r'([-+]?\d+\.\d+)', _line)
            if _m: _tx_lat = float(_m.group(1))
        if 'Site longitude' in _line and _tx_lon is None:
            _m = re.search(r'([-+]?\d+\.\d+)', _line)
            if _m: _tx_lon = float(_m.group(1))

    _hdr_idx = next((i for i, l in enumerate(_lines)
                     if 'Rx Latitude' in l and 'Rx Longitude' in l), None)
    if _hdr_idx is not None:
        _df = pd.read_csv(OFCOM_CSV, skiprows=_hdr_idx, low_memory=False)
        _lat_col  = next((c for c in _df.columns if 'Latitude'    in c), None)
        _lon_col  = next((c for c in _df.columns if 'Longitude'   in c), None)
        _rssi_col = next((c for c in _df.columns if 'measurement' in c or 'dBm' in c), None)
        if _lat_col and _lon_col and _rssi_col:
            _sel = _df[[_lat_col, _lon_col, _rssi_col]].dropna()
            _sel = _sel[pd.to_numeric(_sel[_lat_col],  errors='coerce').notna()]
            _sel = _sel[pd.to_numeric(_sel[_lon_col],  errors='coerce').notna()]
            _sel = _sel[pd.to_numeric(_sel[_rssi_col], errors='coerce').notna()]
            _sel = _sel.astype({_lat_col: float, _lon_col: float, _rssi_col: float})
            if MAX_RX_PLOT:
                _sel = _sel.head(MAX_RX_PLOT)
            _rx_rows = list(zip(_sel[_lon_col], _sel[_lat_col], _sel[_rssi_col]))
            print(f'Loaded {len(_rx_rows)} RX points from CSV')
        else:
            print(f'Could not find lat/lon/RSSI columns. Found: {list(_df.columns[:8])}')

# ── Build figure ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2 if (_rx_rows and PLOT_HEATMAP) else 1,
                         figsize=(18 if (_rx_rows and PLOT_HEATMAP) else 10, 10),
                         dpi=FIG_DPI)
if not isinstance(axes, np.ndarray):
    axes = [axes]

# ── Draw buildings on all axes ───────────────────────────────────────────
_MAT_COLORS = {
    'itu_glass'    : '#aee4f5',
    'itu_brick'    : '#c0704a',
    'itu_concrete' : '#b0b0b0',
    'itu_wood'     : '#c8a87a',
    'itu_metal'    : '#8fa8c0',
}
_DEFAULT_BLD_COLOR = '#d0c8b8'

def _draw_buildings(ax):
    _gdf_plot = gdf_bld.copy()
    # Use WGS84 for plotting (lon/lat axes)
    if hasattr(_gdf_plot, 'crs') and _gdf_plot.crs and str(_gdf_plot.crs) != 'EPSG:4326':
        _gdf_plot = _gdf_plot.to_crs('EPSG:4326')
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        _gdf_plot.plot(ax=ax, facecolor=_DEFAULT_BLD_COLOR, edgecolor='#888888',
                       linewidth=0.15, alpha=0.8)
    ax.set_xlim(SCENE_WEST, SCENE_EAST)
    ax.set_ylim(SCENE_SOUTH, SCENE_NORTH)
    ax.set_aspect('equal')
    ax.set_xlabel('Longitude', fontsize=9)
    ax.set_ylabel('Latitude',  fontsize=9)
    ax.tick_params(labelsize=7)
    # Scene bbox border
    from matplotlib.patches import Rectangle
    _rect = Rectangle((SCENE_WEST, SCENE_SOUTH),
                       SCENE_EAST - SCENE_WEST, SCENE_NORTH - SCENE_SOUTH,
                       linewidth=1.5, edgecolor='black', facecolor='none', linestyle='--')
    ax.add_patch(_rect)

# ── Left panel: buildings + RX dots (plain or single colour) ─────────────
_ax0 = axes[0]
_draw_buildings(_ax0)
_ax0.set_title(f'OSM Buildings + Receiver Locations\n({len(_rx_rows)} RX pts, bbox {(SCENE_EAST-SCENE_WEST)*111:.1f}×{(SCENE_NORTH-SCENE_SOUTH)*111:.1f} km)', fontsize=10)

if _rx_rows:
    _lons_r = [p[0] for p in _rx_rows]
    _lats_r = [p[1] for p in _rx_rows]
    _ax0.scatter(_lons_r, _lats_r, s=2, c='steelblue', alpha=0.5,
                 linewidths=0, zorder=5, label='RX')

if _tx_lon and _tx_lat:
    _ax0.plot(_tx_lon, _tx_lat, marker='*', markersize=14, color='red',
              markeredgecolor='darkred', zorder=10, label='TX')
    _ax0.annotate('TX', (_tx_lon, _tx_lat),
                  textcoords='offset points', xytext=(6, 4),
                  fontsize=8, color='red', fontweight='bold')

_ax0.legend(loc='upper right', fontsize=8, markerscale=3)

# ── Right panel: RSSI heatmap ─────────────────────────────────────────────
if _rx_rows and PLOT_HEATMAP:
    _ax1 = axes[1]
    _draw_buildings(_ax1)
    _ax1.set_title('RSSI Heatmap (measured drive-test)', fontsize=10)

    _lons_h = np.array([p[0] for p in _rx_rows])
    _lats_h = np.array([p[1] for p in _rx_rows])
    _rssi_h = np.array([p[2] for p in _rx_rows])

    _vmin, _vmax = np.percentile(_rssi_h, 2), np.percentile(_rssi_h, 98)

    _sc = _ax1.scatter(_lons_h, _lats_h, c=_rssi_h, s=3,
                       cmap='RdYlGn', vmin=_vmin, vmax=_vmax,
                       alpha=0.75, linewidths=0, zorder=5)
    _cb = fig.colorbar(_sc, ax=_ax1, fraction=0.03, pad=0.02)
    _cb.set_label('RSSI (dBm)', fontsize=9)
    _cb.ax.tick_params(labelsize=8)

    if _tx_lon and _tx_lat:
        _ax1.plot(_tx_lon, _tx_lat, marker='*', markersize=14, color='red',
                  markeredgecolor='darkred', zorder=10)
        _ax1.annotate('TX', (_tx_lon, _tx_lat),
                      textcoords='offset points', xytext=(6, 4),
                      fontsize=8, color='red', fontweight='bold')

plt.tight_layout()

if SAVE_FIG:
    _fig_path = os.path.join(BASE_DIR, 'scene_map.png')
    fig.savefig(_fig_path, dpi=FIG_DPI, bbox_inches='tight')
    print(f'Saved: {_fig_path}')

plt.show()
print('Done.')


## CELL 8 — Interactive 3D RSSI Heatmap (Plotly)

In [ ]:
# ============================================================
# CELL 8 — INTERACTIVE 3D RSSI HEATMAP  (Plotly)
# ============================================================
# Shows:
#   • Interpolated RSSI surface floating above the scene (coverage map)
#   • Raw RSSI scatter dots at ground level coloured by signal strength
#   • Building footprints extruded as 3D grey boxes
#   • TX position as a red star marker
#
# Requirements: pip install plotly scipy
# Run after CELL 4 (gdf_bld loaded) and CELL 7 (_rx_rows loaded).
# Output: interactive HTML saved to BASE_DIR + shown inline.
# ============================================================

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    _HAS_PLOTLY = True
except ImportError:
    _HAS_PLOTLY = False
    print('plotly not installed — run:  pip install plotly')

try:
    from scipy.interpolate import griddata as _griddata
    _HAS_SCIPY = True
except ImportError:
    _HAS_SCIPY = False
    print('scipy not installed — run:  pip install scipy')

if not _HAS_PLOTLY or not _HAS_SCIPY:
    raise SystemExit('Install plotly and scipy first.')

if not _rx_rows:
    raise SystemExit('No RX data — run CELL 7 first to load _rx_rows.')

import numpy as np
import warnings

# ── 1. Coordinate helpers (local metres, centred on scene) ───────────────
_cx = (SCENE_WEST  + SCENE_EAST)  / 2
_cy = (SCENE_SOUTH + SCENE_NORTH) / 2
_lon2m = 111320 * np.cos(np.radians(_cy))   # metres per degree longitude
_lat2m = 111320                               # metres per degree latitude

def _to_local(lon, lat):
    return (lon - _cx) * _lon2m, (lat - _cy) * _lat2m

# ── 2. RX scatter ─────────────────────────────────────────────────────────
_rx_x = np.array([(p[0] - _cx) * _lon2m for p in _rx_rows])
_rx_y = np.array([(p[1] - _cy) * _lat2m for p in _rx_rows])
_rx_z = np.full(len(_rx_rows), 1.5)          # 1.5 m AGL
_rx_r = np.array([p[2] for p in _rx_rows])   # RSSI dBm

_vmin = float(np.percentile(_rx_r, 2))
_vmax = float(np.percentile(_rx_r, 98))

# ── 3. Interpolated RSSI surface ──────────────────────────────────────────
_GRID_N   = 300       # grid resolution (300×300 ≈ ~30 m spacing for 9 km scene)
_SURF_Z   = 20.0      # height of the surface layer (metres above ground)

_x_span = (SCENE_EAST  - SCENE_WEST)  * _lon2m
_y_span = (SCENE_NORTH - SCENE_SOUTH) * _lat2m

_gx = np.linspace(-_x_span/2, _x_span/2, _GRID_N)
_gy = np.linspace(-_y_span/2, _y_span/2, _GRID_N)
_GX, _GY = np.meshgrid(_gx, _gy)

print(f'Interpolating RSSI surface ({_GRID_N}×{_GRID_N}) ...')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _GZ = _griddata(
        np.column_stack([_rx_x, _rx_y]), _rx_r,
        (_GX, _GY), method='linear', fill_value=np.nan
    )
    # Fill remaining NaN holes with nearest-neighbour
    _mask = np.isnan(_GZ)
    if _mask.any():
        _GZ[_mask] = _griddata(
            np.column_stack([_rx_x, _rx_y]), _rx_r,
            (_GX[_mask], _GY[_mask]), method='nearest'
        )
print('Interpolation done.')

# ── 4. Building boxes ─────────────────────────────────────────────────────
print('Building 3D boxes ...')
_bld_traces = []
_MAX_BLDS = 3000   # cap for performance (first N buildings)
_bld_count = 0

_gdf_wgs = gdf_bld.copy()
if hasattr(_gdf_wgs, 'crs') and _gdf_wgs.crs and str(_gdf_wgs.crs) != 'EPSG:4326':
    _gdf_wgs = _gdf_wgs.to_crs('EPSG:4326')

from shapely.geometry import Polygon as _Poly, MultiPolygon as _MPoly

# Collect all building box vertices in a single Mesh3d for speed
_bx_all, _by_all, _bz_all = [], [], []
_bi_all, _bj_all, _bk_all = [], [], []
_v_off = 0

for _, _row in _gdf_wgs.iterrows():
    if _bld_count >= _MAX_BLDS:
        break
    _geom = _row.geometry
    if _geom is None or _geom.is_empty:
        continue
    if isinstance(_geom, _MPoly):
        _geom = max(_geom.geoms, key=lambda g: g.area)
    if not isinstance(_geom, _Poly):
        continue

    # Height
    _h = DEFAULT_HEIGHT_M
    try:
        _hv = float(str(_row.get('height','0')).replace('m','').strip())
        if _hv > 1: _h = float(np.clip(_hv, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M))
    except: pass
    try:
        _lv = float(str(_row.get('building:levels','0')).strip())
        if _lv > 0: _h = float(np.clip(_lv * HEIGHT_PER_LEVEL_M, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M))
    except: pass

    _pts = list(_geom.exterior.coords)
    if len(_pts) < 4:
        continue
    if np.allclose(_pts[0], _pts[-1]):
        _pts = _pts[:-1]
    _n = len(_pts)

    # Bottom ring (z=0) + top ring (z=h)
    for _lon_v, _lat_v in _pts:
        _mx, _my = _to_local(_lon_v, _lat_v)
        _bx_all.append(_mx); _by_all.append(_my); _bz_all.append(0.0)
    for _lon_v, _lat_v in _pts:
        _mx, _my = _to_local(_lon_v, _lat_v)
        _bx_all.append(_mx); _by_all.append(_my); _bz_all.append(_h)

    # Wall faces (two triangles per edge)
    for _i in range(_n):
        _j = (_i + 1) % _n
        _b0, _b1 = _v_off + _i,       _v_off + _j
        _t0, _t1 = _v_off + _n + _i,  _v_off + _n + _j
        _bi_all += [_b0, _b0]; _bj_all += [_b1, _t0]; _bk_all += [_t0, _t1]

    # Flat roof (fan from centroid)
    _rcx = np.mean([p[0] for p in _pts])
    _rcy = np.mean([p[1] for p in _pts])
    _rmx, _rmy = _to_local(_rcx, _rcy)
    _c_idx = _v_off + 2*_n
    _bx_all.append(_rmx); _by_all.append(_rmy); _bz_all.append(_h)
    for _i in range(_n):
        _j = (_i + 1) % _n
        _bi_all.append(_v_off + _n + _i)
        _bj_all.append(_v_off + _n + _j)
        _bk_all.append(_c_idx)

    _v_off += 2*_n + 1
    _bld_count += 1

print(f'  {_bld_count} buildings in 3D mesh')

# ── 5. Assemble Plotly figure ─────────────────────────────────────────────
_fig = go.Figure()

# Buildings mesh
if _bx_all:
    _fig.add_trace(go.Mesh3d(
        x=_bx_all, y=_by_all, z=_bz_all,
        i=_bi_all, j=_bj_all, k=_bk_all,
        color='#c8bfb0', opacity=0.55,
        flatshading=True,
        lighting=dict(ambient=0.7, diffuse=0.5),
        name='Buildings',
        showscale=False,
        hoverinfo='skip',
    ))

# RSSI surface
_fig.add_trace(go.Surface(
    x=_GX, y=_GY,
    z=np.full_like(_GZ, _SURF_Z),   # flat at _SURF_Z height
    surfacecolor=_GZ,
    colorscale='RdYlGn',
    cmin=_vmin, cmax=_vmax,
    opacity=0.72,
    showscale=True,
    colorbar=dict(title='RSSI (dBm)', thickness=15, len=0.6),
    name='RSSI Surface',
    hovertemplate='RSSI: %{surfacecolor:.1f} dBm<extra></extra>',
))

# RX scatter at ground level
_fig.add_trace(go.Scatter3d(
    x=_rx_x, y=_rx_y, z=_rx_z,
    mode='markers',
    marker=dict(
        size=2,
        color=_rx_r,
        colorscale='RdYlGn',
        cmin=_vmin, cmax=_vmax,
        opacity=0.6,
        showscale=False,
    ),
    name='RX measurements',
    hovertemplate='RSSI: %{marker.color:.1f} dBm<extra></extra>',
))

# TX marker
if _tx_lon and _tx_lat:
    _tx_mx, _tx_my = _to_local(_tx_lon, _tx_lat)
    _fig.add_trace(go.Scatter3d(
        x=[_tx_mx], y=[_tx_my], z=[17.0],
        mode='markers+text',
        marker=dict(size=10, color='red', symbol='diamond'),
        text=['TX'], textposition='top center',
        name='Transmitter',
    ))

# Layout
_fig.update_layout(
    title=dict(text='3D RSSI Coverage Map — Nottingham 915 MHz', font=dict(size=14)),
    scene=dict(
        xaxis_title='East–West (m)',
        yaxis_title='North–South (m)',
        zaxis_title='Height (m)',
        aspectmode='manual',
        aspectratio=dict(x=1, y=_y_span/_x_span, z=0.08),
        camera=dict(eye=dict(x=0, y=-1.6, z=0.9)),
        bgcolor='#f0f0f0',
    ),
    margin=dict(l=0, r=0, t=40, b=0),
    legend=dict(x=0.01, y=0.99, font=dict(size=10)),
    width=1200, height=750,
)

# ── 6. Save HTML + show ───────────────────────────────────────────────────
_html_path = os.path.join(BASE_DIR, 'rssi_3d_heatmap.html')
_fig.write_html(_html_path, include_plotlyjs='cdn')
print(f'Saved interactive HTML: {_html_path}')
print('Open in any browser — rotate, zoom, hover for RSSI values.')

_fig.show()
